# The round trip without a notebook: export from the web app, read in one call

<a href="https://colab.research.google.com/github/theislab/cellucid-python/blob/main/docs/user_guide/python_package/f_notebooks_tutorials/44_round_trip_without_a_notebook.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

No notebook is open while you work: you (or a colleague) open the
dataset at www.cellucid.com, make highlights, labels and gene sets,
and export them. Back in Python, one call reads every export onto the
AnnData: `cellucid.pull(adata, path)`.

**Time:** 10 minutes. The same tutorial in R:
[Without a notebook, in R](../../r_package/f_notebooks_tutorials/10_round_trip_without_a_notebook.ipynb).

In [ ]:
# A fresh runtime (Google Colab, a new environment) installs Cellucid once.
import importlib.util

if importlib.util.find_spec("cellucid") is None:
    %pip install --quiet cellucid

## The data

The 2,531-cell mouse pancreas (endocrinogenesis, day 15.5) AnnData checked
into the cellucid-python repository: UMAP in `obsm['X_umap_2d']`, cell types
in `obs['clusters']`, 27,998 genes. In a clone of the repository the cell
below finds it; anywhere else (Colab) it downloads it once. Use your own
`adata` in its place.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import anndata as ad

import cellucid

NAME = "endocrinogenesis_day15.5_raw.h5ad"
URL = (
    "https://raw.githubusercontent.com/theislab/cellucid-python/main/"
    "docs/user_guide/python_package/f_notebooks_tutorials/datasets/" + NAME
)


def pancreas_h5ad() -> Path:
    """The Pancreas file of a cellucid-python clone, else a copy downloaded once (35 MB)."""
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for candidate in (base / "datasets" / NAME, base / "docs/user_guide/python_package/f_notebooks_tutorials/datasets" / NAME):
            if candidate.is_file():
                return candidate
    if not (here / NAME).is_file():
        urlretrieve(URL, here / NAME)
    return here / NAME


DATA_PATH = pancreas_h5ad()
adata = ad.read_h5ad(DATA_PATH)
adata

## Give the file an id first

A saved session belongs to one dataset, named by its id. A file that
states its own id in `uns['cellucid']['id']` has the same id however
it is opened, so write one in before you share it. (A file without
one is named after its file name in the web app:
`h5ad_<file name>`.)

In [ ]:
adata.uns["cellucid"] = {"id": "pancreas-e15-5", "name": "Pancreas E15.5"}
adata.write_h5ad("pancreas.h5ad")

## In the web app

Open https://www.cellucid.com, then **Session › Open… › File…** and
choose `pancreas.h5ad` (nothing is uploaded: the browser reads the
file). Then make and export:

| Make | Export | File |
|---|---|---|
| a lasso or any highlight (**Highlighting**, **Confirm**) | **CSV** beside **Clear** | `<page>_cells.csv`: `obs_name`, then one `true`/`false` column per group |
| a label field (**Copy** `clusters`, rename it and its categories) | the field's **Export** button | `<field>.csv`: `obs_name`, then the field |
| a gene set (**Analysis › Gene signature**, the gene set editor) | the editor's **Export** | a `.gmt` file |
| everything at once | **Session › Save** | a `.cellucid-session` file |
| the highlighted cells as an AnnData | **.h5ad** beside **Clear** | `<page>_cells.h5ad` ([re-analyse it](46_export_an_h5ad_subset_and_reanalyse.ipynb)) |

Put the files in one folder (here `cellucid-exports` next to this
notebook; in Colab, upload them with the Files pane).

In [ ]:
EXPORTS = Path("cellucid-exports")

adata = ad.read_h5ad("pancreas.h5ad")
summary = cellucid.pull(adata, EXPORTS)
summary.columns

`pull` takes one file, a list of files, or a folder (it reads every
`.csv`, `.gmt` and `.cellucid-session` in it):

- A **CSV** column is written under its own name. `true`/`false` only
  is a highlight (`bool`; cells the file does not list are `False`),
  numbers a continuous column, anything else a categorical one. A
  cell the file names that `adata` does not have is refused, by name.
- A **GMT** file's sets go to `adata.uns['cellucid']['gene_sets']`.
- A **session** writes its highlight groups and label fields as the
  live pull does, after checking it was saved on this dataset (the id
  in `uns['cellucid']['id']`; or pass `expected_dataset_id=`).

Pulling the same folder again replaces the columns it wrote. A column
of the same name that Cellucid did not write (`clusters`, say) is a
conflict: export your own fields, or pass `on_conflict="suffix"`.

In [ ]:
print(adata.obs[list(summary.columns)].head())
print(summary.gene_sets)

A GMT file is also read on its own: `cellucid.read_gmt(path)` returns
`{set name: [genes]}`.

In [ ]:
for path in sorted(EXPORTS.glob("*.gmt")):
    print(path.name, cellucid.read_gmt(path))

## Next

- [Export a subset and re-analyse it](46_export_an_h5ad_subset_and_reanalyse.ipynb)
- [Gene sets round trip](45_gene_sets_round_trip.ipynb)
- Reference: [the web app's exports](../g_api_reference_coverage/api/round_trip.md)
- R: [Without a notebook, in R](../../r_package/f_notebooks_tutorials/10_round_trip_without_a_notebook.ipynb)